# Residential mobility and social structure

**How many times do people move, and who moves repeatedly?**

This notebook analyses the binary outcome *hypermobile vs. low-mobile* in the
«Социальная структура» survey (09.03.2025, **N = 7,563**). It was written for the
research group **«Городская повседневность»** (Urban Everyday Life).

**Contents**

1. Setup
2. Data preparation
3. Outcome: mobility typology
4. Cleaning
5. Descriptives
6. Bivariate associations
7. Factor analysis of attitudes (blocks B107 and B114)
8. Cluster analysis
9. Logistic regression with VIF diagnostics

**Companion file.** `../r/residential_mobility.R` re-implements the exploratory
stage in R (`gtsummary`, `vcd`, `FactoMineR`, Gower clustering).

**Data are not distributed with this repository.** See the README for the schema
of the expected input file and the ethics note.

## 1. Setup

In [ ]:
# Only the packages actually used below are imported.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyreadstat
import statsmodels.api as sm
from statsmodels.formula.api import logit
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF
from patsy import dmatrix
from factor_analyzer import FactorAnalyzer
from scipy.stats import kruskal, ttest_ind
from scipy.cluster.hierarchy import linkage, dendrogram, cut_tree
from sklearn.preprocessing import MinMaxScaler
from stargazer.stargazer import Stargazer
from stargazer.utils import LogitOdds
from pysummaries import get_table_summary, pandas_to_report_html

sns.set_theme(style="white", palette="deep")
pd.set_option("display.min_rows", 10)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

# The SPSS file is not part of this repository (see README, "Data and ethics").
DATA_PATH = "Social Structure_09_03_2025_itog.sav"

# More than this many moves since age 16 counts as "hypermobile".
# The companion R script uses `>= 3`; see README, "Known inconsistencies".
MOVES_THRESHOLD = 3

## 2. Data preparation

The SPSS file stores questions under technical names (`Q6`, `Q115`, …) and keeps
the wording separately in the metadata. The mapping below translates the labels
into short, code-friendly names; the value labels (`1 = Да`, `2 = Нет`, …) are
recovered from the file metadata rather than hard-coded.

In [ ]:
VARIABLE_MAP = {
    "В115. Возраст: число лет": "age",
    "В117. Пол": "sex",
    "В6. Общее число переездов, начиная с 16 лет": "all_reloc",
    "В7. Число переездов в пределах одного и того же населённого пункта?": "city_reloc",
    "В8. Количество переездов за пределы  населённого пункта\\в пределах России": "russia_reloc",
    "В8. Количество переездов за пределы  населённого пункта\\в пределах СССР (до распада СССР).": "ussr_reloc",
    "В9. Количество переездов в другую страну или из другой страны.": "world_reloc",
    "В13. Уровень образования респондента": "educ",
    "В19. Работаете ли Вы в настоящее время?": "work",
    "В25. Как оформлены отношения с нанимателем": "contract",
    "В87. Совокупный месячный доходреспондента": "personal_income",
    "В88. Совокупный месячный доход семьи респондента": "fam_income",
    "В61. Есть ли у Вас дети?": "children",
    "В60. Семейное положение респондента": "marrige",
    "В94. Жилищные условия респондента": "home",
    "В95. Кто собственник  жилья респондента": "home_owner",
    "В93. Удовлетворённость материальным положением": "fin_satisfaction",
    "В91. К какой из следующих групп Вы себя относите?": "fin_group",
    "В92. Как изменится материальное положение респондента в будущем 2025 году": "fin_pred",
    "В101. Вы – верующий человек?": "relig",
    "В109. По Вашему мнению, у Вас достаточно воли, чтобы, если того потребует ситуация, радикально изменить свою жизнь?": "volya",
    "В119. Укажите, в каком типе населенного пункта Вы живете большую часть года.": "city",
    "В21. Профессия респондента в настоящее время: укрупнённые группы ISCO": "work_name",
    "В21. Кодировка работы в настоящее время: ISCO-08": "work_name2",
}

In [ ]:
spss, meta = pyreadstat.read_sav(DATA_PATH, apply_value_formats=False)

df = spss.copy()
df.columns = meta.column_labels                 # technical names -> question wording
df = df.rename(columns=VARIABLE_MAP)            # question wording -> short names

# Rebuild the value labels under the short names:
#   column name -> question wording -> short name -> {code: label}
column_to_short = {
    column: VARIABLE_MAP[label]
    for column, label in meta.column_names_to_labels.items()
    if label in VARIABLE_MAP
}
VALUE_LABELS = {
    column_to_short[column]: labels
    for column, labels in meta.variable_value_labels.items()
    if column in column_to_short
}

print(f"Loaded {df.shape[0]:,} respondents × {df.shape[1]} variables")
print(f"Value labels recovered for {len(VALUE_LABELS)} of {len(VARIABLE_MAP)} mapped variables")
df[list(VARIABLE_MAP.values())].head(3)

## 3. Outcome: mobility typology

The outcome is **binary**: a respondent is *hypermobile* if they report more than
three moves since the age of 16. The raw count has a long right tail (maximum 99
moves), so the distribution is shown twice — once raw and once restricted to the
bulk of the data by the 1.5 × IQR rule.

In [ ]:
moves_summary = df["all_reloc"].describe().to_frame("all_reloc")
sns.kdeplot(df["all_reloc"]).set(
    title="Distribution of the number of moves since age 16",
    xlabel="number of moves",
)
display(moves_summary)

In [ ]:
q1, q3 = df["all_reloc"].quantile([0.25, 0.75])
iqr_upper = q3 + 1.5 * (q3 - q1)
within_iqr = df["all_reloc"] < iqr_upper

sns.histplot(df.loc[within_iqr, "all_reloc"]).set(
    title=f"Moves below the 1.5 × IQR threshold ({iqr_upper:.0f})",
    xlabel="number of moves",
)
display(df.loc[within_iqr, "all_reloc"].describe().to_frame())

In [ ]:
df["type"] = np.where(df["all_reloc"] > MOVES_THRESHOLD, "Сверхмобильные", "Маломобильные")
df["type_bin"] = (df["type"] == "Сверхмобильные").astype(int)

display(df["type"].value_counts().to_frame("n"))

In [ ]:
sns.histplot(df.loc[df["type_bin"] == 1, "all_reloc"]).set(
    title="Number of moves among the hypermobile group",
    xlabel="number of moves",
)
display(df.loc[df["type_bin"] == 1, "all_reloc"].describe().to_frame("Сверхмобильные"))

## 4. Cleaning

Two cleaning steps are applied before any modelling.

* **Implausible incomes.** Monthly incomes above 10 million roubles are treated
  as data-entry errors and dropped.
* **Missing-value codes.** The questionnaire codes "difficult to answer" as `9`
  (and "other" as `8` for housing tenure). These are not substantive categories
  and are recoded to `NaN`, so that they cannot be read as an ordinal position.

In [ ]:
INCOME_CEILING = 10_000_000

for column in ["personal_income", "fam_income"]:
    implausible = df[column] > INCOME_CEILING
    print(f"{column}: dropping {implausible.sum()} implausible value(s)")
    display(df.loc[implausible, ["personal_income", "fam_income", "fin_group"]])
    df = df.loc[~implausible].copy()

print(f"Rows after cleaning: {len(df):,}")

In [ ]:
# SPSS missing-value codes, by variable.
MISSING_CODES = {
    "fin_satisfaction": [9],      # 9 = "затрудняюсь ответить"
    "fin_pred": [9],
    "volya": [9],
    "home": [10],                 # 10 = "другое"
    "home_owner": [8, 9],         # 8 = "другое", 9 = "затрудняюсь ответить"
}

for column, codes in MISSING_CODES.items():
    df[column] = df[column].mask(df[column].isin(codes))

display(df[list(MISSING_CODES)].isna().sum().to_frame("n missing"))

In [ ]:
# Owner-occupancy flag: 1 if the respondent (or their family) owns the dwelling.
#
# Correction relative to the original analysis: the missing codes were mapped to
# 0, which merged "did not answer" into "does not own the dwelling". Missing
# values are preserved here, so the flag has three states (1 / 0 / NaN) and the
# models below use completed cases only.
owns = (df["home_owner"] == 1).astype(float)
df["home_owner_bin"] = np.where(df["home_owner"].isna(), np.nan, owns)

ownership_table = sm.stats.Table(pd.crosstab(df["home_owner_bin"], df["type"]))
display(pandas_to_report_html(
    ownership_table.standardized_resids.round(2),
    caption=f"Standardised residuals; chi-square p = "
            f"{ownership_table.test_nominal_association().pvalue:.2f}",
))

## 5. Descriptives

Standard "table one": categorical variables as n (%), numeric variables as
mean (SD), median [IQR], range and share missing, split by mobility type.
Categorical columns are mapped to their text labels before tabulation.

In [ ]:
# Categorical variables whose SPSS codes are replaced by their text labels.
CATEGORICAL_LABELLED = ["relig", "sex", "children", "work", "marrige", "educ"]

# `home_owner` is nominal (1 = the family owns the dwelling, 2 = the state,
# 5 = renting, …) but is kept on its numeric scale here so that the cluster
# analysis below reproduces the original specification exactly. Treating a
# nominal variable as ordinal in a distance-based method is not ideal — see
# README, "Known inconsistencies", for the suggested fix.
NUMERICAL = ["home_owner", "age", "personal_income", "fam_income", "volya",
             "fin_pred", "fin_satisfaction"]
FLAGS = ["home_owner_bin"]


def with_labels(frame, columns):
    """Replace SPSS numeric codes with their text labels."""
    out = frame[columns].copy()
    for column in columns:
        labels = VALUE_LABELS.get(column)
        if labels:
            out[column] = out[column].map(labels).astype("category")
    return out


stat = pd.concat([
    with_labels(df, CATEGORICAL_LABELLED),
    df[NUMERICAL + FLAGS],
    df["type"],
], axis=1)

summary_table = get_table_summary(
    stat,
    strata="type",
    categorical_functions="n_percent",
    numerical_functions="meansd_medianiqr_minmax_missing",
)
summary_table

## 6. Bivariate associations

For every categorical or ordinal variable, the share of hypermobile respondents
is plotted as bars (with a trend line for ordered variables), next to the raw
distribution of the variable across the two mobility types.

In [ ]:
def mobility_by_group(variable, ordered=False, data=None, min_count=0):
    """Share of hypermobile respondents by `variable`.

    Returns a reportable table and draws two panels: the share of hypermobile
    respondents per category, and the distribution of `variable` by mobility type.
    """
    data = df if data is None else data

    report = (
        data.groupby(variable)["type_bin"]
        .agg(count="count", share=lambda s: s.mean() * 100, std="std")
        .reset_index()
    )
    labels = VALUE_LABELS.get(variable)
    report["label"] = report[variable].map(labels) if labels else report[variable]
    report = report[report["count"] >= min_count].round(3)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.barplot(report, x=variable, y="share", ax=axes[0], color="grey")
    if ordered:
        sns.lineplot(report, x=variable, y="share", ax=axes[0], color="black")
    axes[0].set_xticks(range(len(report)))
    axes[0].set_xticklabels(report["label"])
    axes[0].set(xlabel=variable, ylabel="% hypermobile")
    sns.pointplot(data, x="type", y=variable, color="black", ax=axes[1])
    plt.tight_layout()

    return pandas_to_report_html(
        report.rename(columns={"share": "% Сверхмобильных"}),
        caption=variable,
        footer=f"Количество ответивших: {report['count'].sum()}",
    )

In [ ]:
mobility_by_group("city", ordered=True)

In [ ]:
mobility_by_group("relig")

In [ ]:
mobility_by_group("educ", ordered=True)

In [ ]:
mobility_by_group("fin_satisfaction", ordered=True)

In [ ]:
mobility_by_group("fin_pred", ordered=True)

In [ ]:
mobility_by_group("fin_group", ordered=True)

In [ ]:
mobility_by_group("volya", ordered=True)

In [ ]:
mobility_by_group("home", ordered=True)

In [ ]:
mobility_by_group("home_owner")

In [ ]:
mobility_by_group("children")

In [ ]:
mobility_by_group("work")

In [ ]:
mobility_by_group("work_name")

In [ ]:
# Occupation at the level of ISCO-08 unit groups, restricted to groups with a
# reasonable number of respondents.
occupation = (
    df.groupby("work_name2")["type_bin"]
    .agg(count="count", share=lambda s: s.mean() * 100, std="std")
    .reset_index()
)
occupation["label"] = occupation["work_name2"].map(VALUE_LABELS.get("work_name2", {})).fillna("(no label)")
occupation = occupation[occupation["count"] > 15].sort_values("share", ascending=False)

display(pandas_to_report_html(
    occupation.rename(columns={"share": "% Сверхмобильных"}).round(2).head(10),
    caption="Профессия (ISCO-08, группы с n > 15)",
))

In [ ]:
sns.catplot(stat, y="personal_income", x="type")

In [ ]:
# Income has a heavy right tail, so the boxplot is drawn on the IQR-trimmed
# subsample (see also `outliers()` in the R companion script).
income_iqr = (stat["personal_income"].quantile(0.75) - stat["personal_income"].quantile(0.25)) * 1.5
income_cutoff = stat["personal_income"] < stat["personal_income"].quantile(0.75) + income_iqr
income_trimmed = stat[income_cutoff]

sns.boxplot(income_trimmed, x="type", y="personal_income").set(
    title="Personal income by mobility type (IQR-trimmed)",
)

In [ ]:
low = income_trimmed.loc[income_trimmed["type"] == "Маломобильные", "personal_income"]
high = income_trimmed.loc[income_trimmed["type"] == "Сверхмобильные", "personal_income"]
t_statistic, p_value = ttest_ind(low, high)

print(f"Two-sample t-test on IQR-trimmed personal income")
print(f"  n = {len(low):,} vs {len(high):,}")
print(f"  t = {t_statistic:.3f}, p = {p_value:.4f}")

## 7. Factor analysis of attitudes

Two batteries of Likert items are reduced to orthogonal factors with a principal
component extraction:

* **B107** — 17 items on the causes of poverty, fairness and the role of
  authorities. Five factors.
* **B114** — 19 items on individualism, collectivism and family obligation.
  Four factors.

Code `9` ("difficult to answer") is set to `NaN`; items are min-max scaled to a
common 0–1 range before extraction, and complete cases are used. Loadings below
|0.4| are blanked in the printed matrices so that the factor pattern stays
readable.

In [ ]:
def factor_analysis(items, n_factors, names, title):
    """Principal-component factor analysis of a Likert battery.

    Returns the fitted analyser, the loading matrix, communalities and the
    factor scores (aligned to the original index, with NaN for incomplete cases).
    """
    complete = df[items].replace(9, np.nan).dropna()
    scaled = pd.DataFrame(
        MinMaxScaler().fit_transform(complete),
        index=complete.index,
        columns=items,
    )

    analyser = FactorAnalyzer(n_factors=n_factors, method="principal").fit(scaled)

    eigenvalues, _, cumulative = analyser.get_factor_variance()
    print(f"Eigenvalues: {np.round(eigenvalues, 3)}")
    print(f"Explained variance (cumulative): {cumulative[-1]:.3f}")
    print(f"Complete cases: {len(complete):,}")

    sns.pointplot(analyser.get_eigenvalues()[0]).set(title=f"Scree plot — {title}")

    loadings = pd.DataFrame(analyser.loadings_, index=items, columns=names)
    communalities = (
        pd.DataFrame(analyser.get_communalities(), index=items, columns=["communality"])
        .sort_values("communality", ascending=False)
    )
    scores = pd.DataFrame(analyser.transform(scaled), index=complete.index, columns=names)

    return analyser, loadings, communalities, scores


def pattern(loadings, cutoff=0.4):
    """Blank loadings below |cutoff| to make the factor pattern readable.

    Correction: the B114 version of this filter originally read
    `(x < cutoff) or (x < -cutoff)`, which evaluates to `x < cutoff` and
    therefore hid every loading below -0.4. The two blocks now share this
    absolute-value filter, so large negative loadings are shown.
    """
    return loadings.round(3).mask(loadings.abs() < cutoff).fillna("")

In [ ]:
B107_ITEMS = [
    "В107.1. Наша жизнь зависит от внешних обстоятельств",
    "В107.2. В нашем обществе среди управленцев слишком мало женщин",
    "В107.3. Рабочие сами могут управлять производством без начальников",
    "В107.4. Причина бедности – в том, что у многих людей не хватает ума, чтобы утверждать себя в современном обществе",
    "В107.5. Причина бедности в том, что многие люди просто не хотят работать",
    "В107.6. Причина бедности в низкой оплате труда большинства работников",
    "В107.7. Причина бедности в том, что закрылись предприятия, людям негде работать",
    "В107.8. В справедливом обществе у всех людей – одинаковый жизненный уровень",
    "В107.9. В справедливом обществе трудолюбивые люди живут лучше, чем ленивые",
    "В107.10. В справедливом обществе образованные люди живут лучше, чем необразованные",
    "В107.11. В справедливом обществе у каждого гражданина есть право на бесплатное образование и медицинское обслуживание",
    "В107.12. Каждый человек обязан приложить все силы, чтобы дать своим детям хорошее образование",
    "В107.15. Выбрать правильный путь для страны в этом сложном мире могут только руководители и специалисты",
    "В107.16. Большинство людей не понимает, в чём состоят их истинные интересы",
    "В107.17. Несколько сильных руководителей могут сделать для страны больше, чем все законы и все разговоры",
    "В107.13. Для меня интересная работа важнее, чем деньги",
    "В107.14. Интересной работой можно пожертвовать, если есть возможность хорошо заработать",
]
B107_NAMES = ["sov_left", "lib_cons", "prog_left", "money", "merit"]

In [ ]:
fa107, loadings107, communalities107, scores107 = factor_analysis(
    B107_ITEMS, n_factors=5, names=B107_NAMES, title="block B107",
)
display(pattern(loadings107))
display(communalities107)
df = df.join(scores107)

In [ ]:
B114_ITEMS = [
    "В114.1. Мне нравится выделяться среди окружающих",
    "В114.2. Только я несу ответственность за всё, что со мной происходит",
    "В114.3. Принимая решения, я не беспокоюсь о том, что о них подумают другие",
    "В114.4. В трудных ситуациях я полагаюсь только на свои силы",
    "В114.5. Меня огорчает, когда кто-то оказывается успешнее меня",
    "В114.6. Меня раздражает, когда со мной спорят об очевидных вещах",
    "В114.7. Для меня коллективный результат важнее, чем мой личный успех",
    "В114.8. В жизни цель всегда оправдывает средства",
    "В114.9. Для меня важно превосходить других в том, что я делаю",
    "В114.10. В случае необходимости я пожертвую своими интересами ради благополучия членов моей семьи",
    "В114.11. Дети могут доверить уход за пожилыми родителями социальным работникам",
    "В114.12. Я выбрал(а) профессию с учётом мнения моих родителей",
    "В114.13. Детей нужно приучать жить по принципу: «Делу – время, потехе – час»",
    "В114.14. Я радуюсь, когда мои коллеги добиваются успеха",
    "В114.15. Считаю, что участие и забота о родных важнее финансовой поддержки",
    "В114.16. Для меня очень важно иметь дружеские отношения с коллегами",
    "В114.17. Я уверен(на), что все проблемы, возникающие между соседями, можно решить, найдя компромисс",
    "В114.18. Принимая важные решения, я советуюсь с близкими людьми",
    "В114.19. Я прежде всего думаю о благополучии коллектива, а потом уже о собственных интересах",
]
B114_NAMES = ["emot", "ego", "colect", "ind"]

In [ ]:
fa114, loadings114, communalities114, scores114 = factor_analysis(
    B114_ITEMS, n_factors=4, names=B114_NAMES, title="block B114",
)
display(pattern(loadings114))
display(communalities114)
df = df.join(scores114)

In [ ]:
def compare_factors(factor_names):
    """Group means and independent-samples t-tests for a block of factor scores.

    Complete cases are taken across the whole block, so every factor in the
    block is tested on the same respondents.
    """
    means = df.groupby("type_bin")[factor_names].mean().T
    means.columns = ["0 — маломобильные", "1 — сверхмобильные"]

    hypermobile = df.loc[df["type_bin"] == 1, factor_names].dropna()
    low_mobile = df.loc[df["type_bin"] == 0, factor_names].dropna()
    test = ttest_ind(hypermobile, low_mobile)

    tests = pd.DataFrame(
        {
            "t": np.round(test.statistic, 3),
            "p_value": np.round(test.pvalue, 4),
            "df": np.round(test.df, 1),
        },
        index=factor_names,
    )
    return means.round(3), tests


display(*compare_factors(B107_NAMES))
display(*compare_factors(B114_NAMES))

## 8. Cluster analysis

Agglomerative clustering (complete linkage, city-block distance) over the
sociodemographic block. Numeric variables are min-max scaled and categorical
variables are one-hot encoded, so that no single variable dominates the distance
matrix. Two runs are reported: the hypermobile subgroup alone, and the full
sample. Differences between clusters are tested with Kruskal–Wallis for numeric
variables and chi-square for categorical ones.

In [ ]:
def cluster_features(frame):
    """Min-max scale the numeric columns and one-hot encode the categorical ones.

    Dummy columns are named `variable::value`. The `::` separator matters:
    `home_owner` is a prefix of `home_owner_bin`, so a `startswith(f"{col}_")`
    match would silently merge the dummy sets of the two variables.
    """
    numeric = frame.select_dtypes("float64")
    scaled = pd.DataFrame(
        MinMaxScaler().fit_transform(numeric),
        index=numeric.index,
        columns=numeric.columns,
    )

    dummies = {}
    for column in frame.select_dtypes("category").columns:
        for value, indicator in pd.get_dummies(frame[column]).items():
            dummies[f"{column}::{value}"] = indicator

    return pd.concat([scaled, pd.DataFrame(dummies, index=frame.index)], axis=1)


def dummies_to_labels(features, frame):
    """Collapse one-hot blocks back into one labelled column per variable."""
    out = pd.DataFrame(index=features.index)
    for column in frame.select_dtypes("category").columns:
        block = features[[c for c in features.columns if c.startswith(f"{column}::")]]
        out[column] = block.idxmax(axis=1).str.split("::").str[-1]
    return out


def run_cluster_analysis(frame, n_clusters=4, title=""):
    """Cluster `frame`, draw a dendrogram, profile the clusters and test them."""
    features = cluster_features(frame)

    linkage_matrix = linkage(features, method="complete", metric="cityblock")
    dendrogram(linkage_matrix)
    plt.title(f"Dendrogram — {title}")
    plt.show()

    clusters = pd.Series(
        cut_tree(linkage_matrix, n_clusters=n_clusters).ravel(),
        index=features.index,
        name="cluster",
    )
    print(f"Cluster sizes (k = {n_clusters}):")
    display(clusters.value_counts().sort_index().to_frame("n"))

    labelled = pd.concat(
        [frame.select_dtypes("float64"), dummies_to_labels(features, frame)],
        axis=1,
    )
    labelled["cluster"] = clusters

    display(get_table_summary(
        labelled, strata="cluster",
        numerical_functions="meansd_medianiqr_minmax_missing",
    ))

    tests = []
    for column in labelled.columns.drop("cluster"):
        if labelled[column].dtype == "float64":
            statistic, p_value = kruskal(
                *[labelled.loc[labelled["cluster"] == k, column] for k in range(n_clusters)]
            )
            tests.append({"variable": column, "test": "Kruskal-Wallis",
                          "statistic": round(statistic, 2), "p_value": round(p_value, 3)})
        else:
            table = sm.stats.Table(pd.crosstab(labelled[column], labelled["cluster"]))
            association = table.test_nominal_association()
            tests.append({"variable": column, "test": "Chi-square",
                          "statistic": round(association.statistic, 2),
                          "p_value": round(association.pvalue, 3)})

    display(pandas_to_report_html(pd.DataFrame(tests).set_index("variable")))
    return labelled

In [ ]:
cluster_stat = stat.dropna().copy()
cluster_stat["home_owner_bin"] = pd.Categorical(
    cluster_stat["home_owner_bin"].map({1.0: "Является собственником", 0.0: "Не является собственником"}),
    categories=["Не является собственником", "Является собственником"],
)

In [ ]:
hypermobile_stat = cluster_stat.loc[cluster_stat["type"] == "Сверхмобильные"]
display(pandas_to_report_html(
    pd.DataFrame({"Переменные для кластерного анализа": stat.columns}),
    show_index=False,
))
labelled_hypermobile = run_cluster_analysis(
    hypermobile_stat.drop(columns="type"),
    n_clusters=4,
    title="hypermobile respondents",
)

In [ ]:
labelled_full = run_cluster_analysis(
    cluster_stat.drop(columns="type"),
    n_clusters=4,
    title="full sample",
)

## 9. Logistic regression

The probability of being hypermobile is modelled as a function of
sociodemographic characteristics. Unlike the original version, the estimation
sample is built explicitly (complete cases only) so that the reported N matches
the coefficients, and the variance inflation factors are computed on the design
matrix that the model actually uses.

In [ ]:
MODEL_VARIABLES = ["relig", "sex", "children", "work", "home_owner_bin", "age",
                   "educ", "personal_income", "volya", "fin_pred",
                   "fin_satisfaction", "fam_income"]

MODEL_FORMULA = (
    "type_bin ~ C(relig) + C(sex) + C(children) + C(work) + C(home_owner_bin)"
    " + age + educ + personal_income + volya + fin_pred + fin_satisfaction + fam_income"
)

# Complete-case estimation: the analysis sample is built explicitly and reported,
# instead of relying on statsmodels to drop rows silently.
model_data = df[MODEL_VARIABLES + ["type_bin"]].dropna()
print(f"Complete cases: {len(model_data):,} of {len(df):,}")

model = logit(MODEL_FORMULA, data=model_data).fit(disp=False)
display(Stargazer([LogitOdds(model)]))

In [ ]:
odds_ratios = pd.DataFrame({
    "exp(B)": np.exp(model.params),
    "p_value": model.pvalues,
}).drop(index="Intercept").sort_values("p_value")

odds_ratios.index.name = "var"
display(pandas_to_report_html(odds_ratios.round(3)))

In [ ]:
def forest_plot(model, title="Odds ratios with 95% confidence intervals"):
    """Forest plot on the odds-ratio scale.

    Correction: the original function plotted the raw log-odds coefficients with
    symmetric ±1.96 SE bars, while the accompanying table reported odds ratios —
    the two panels were therefore on different scales. Both now show odds ratios
    with asymmetric confidence intervals, on a log axis.
    """
    params = model.params.drop("Intercept")
    confidence = model.conf_int().drop("Intercept")

    odds_ratio = np.exp(params)
    lower = np.exp(confidence[0])
    upper = np.exp(confidence[1])

    figure, axes = plt.subplots(figsize=(7, 0.45 * len(params) + 2))
    y = np.arange(len(params))
    axes.errorbar(
        odds_ratio, y,
        xerr=[odds_ratio - lower, upper - odds_ratio],
        fmt="o", markersize=7, color="black", ecolor="grey", capsize=3,
    )
    axes.axvline(1, linestyle="--", color="grey")
    axes.set_yticks(y)
    axes.set_yticklabels(params.index)
    axes.set(xscale="log", xlabel="Odds ratio (log scale)", title=title)
    axes.invert_yaxis()
    plt.tight_layout()


forest_plot(model)

In [ ]:
# VIF on the actual design matrix — including the dummy variables — rather than
# on the raw columns, which correspond to no estimated coefficient. Note that
# the dummy set of a multi-level factor is expected to show elevated VIF; a
# generalised VIF (GVIF) is the appropriate diagnostic for those.
design = dmatrix(MODEL_FORMULA.split("~")[1], data=model_data, return_type="dataframe")

vif_table = pd.DataFrame(
    {"VIF": [VIF(design.values, i) for i in range(design.shape[1])]},
    index=design.columns,
)
vif_table = (
    vif_table.drop(index="Intercept", errors="ignore")
    .sort_values("VIF", ascending=False)
    .round(2)
)
vif_table["flag"] = np.where(vif_table["VIF"] > 10, "*", "")

display(pandas_to_report_html(
    vif_table,
    footer="* VIF > 10 — possible multicollinearity. Dummy sets for a "
           "multi-level factor legitimately show high VIF.",
))

---

### Notes on what changed

This notebook is a readability refactor of the original analysis script. The
statistical specification is unchanged except where a genuine defect was found:

| Change | Effect on results |
| --- | --- |
| Duplicate `analysis('fin_pred')` call removed | none |
| Broken `from_dummy()` helper replaced by `dummies_to_labels()` | none — the helper was dead code |
| Duplicated cluster block extracted into `run_cluster_analysis()` | none |
| `cut_tree(...).ravel()` — avoids relying on implicit 2-D broadcasting | none |
| B114 loading filter `or` → absolute value via `pattern()` | **shows loadings below −0.4 that were previously hidden** |
| `home_owner_bin`: missing codes no longer mapped to 0 | **changes the logistic regression sample and the ownership coefficient** |
| Forest plot moved to the odds-ratio scale | cosmetic, but the figure now matches the table |
| VIF computed on the design matrix instead of raw columns | **different (and meaningful) VIF values** |

Because outputs are cleared in this notebook, re-run it against the survey file
to regenerate every table and figure. The numbers quoted in `../RESULTS.md` come
from the original run and predate the three corrections marked in bold.